In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
plt.style.use('default')
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False 
import pickle
from sklearn.preprocessing import StandardScaler
from itertools import combinations_with_replacement
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
cm = 1/2.54
# 字号
fontsize = 8
seed = 2025

In [25]:
# 模型集合：随机森林、梯度提升树、LightGBM、AdaBoost、支持向量机、神经网络
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
models = {
    'RF': RandomForestRegressor(random_state=seed),
    'GBDT': GradientBoostingRegressor(random_state=seed),
    'LGBM': LGBMRegressor(random_state=seed),
    'AdaBoost': AdaBoostRegressor(random_state=seed),
    'SVR': SVR(),
    'NN': MLPRegressor(random_state=seed),
    'XGBoost': xgb.XGBRegressor(random_state=seed)
}
def get_data(data):
    x = data.iloc[:, :6]
    y = data.iloc[:, -2]/3500
    return x, y

### CC阶段

In [26]:
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_CC.pkl', 'rb') as file:
    train, val, test = pickle.load(file)
train = pd.concat([train, val], axis=0).reset_index(drop=True)
# 特征添加上工况,id以-_#为分割的第0-2部分转换为独热编码
def add_feature(data):
    data['temp'] = data['id'].apply(lambda x: x.split('-')[0])
    data['charge_rate'] = data['id'].apply(lambda x: x.split('-')[1].split('_')[0])
    data['discharge_rate'] = data['id'].apply(lambda x: x.split('_')[1].split('-')[0])
    data['temp'] = data['temp'].astype(int)
    data['charge_rate'] = data['charge_rate'].astype(int)
    data['discharge_rate'] = data['discharge_rate'].astype(int)
    # 按6, 7, 8, 0, 1, 2, 3, 4, 5列的顺序排列
    data = data.iloc[:, [6, 7, 8, 0, 1, 2, 3, 4, 5]]
    return data
train = add_feature(train)
test = add_feature(test)
# 归一化
scaler = StandardScaler()
train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
test.iloc[:, :-3] = scaler.transform(test.iloc[:, :-3])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\1711806692.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.68765121 -1.68765121 -1.68765121 ...  0.64009949  0.64009949
  0.64009949]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\1711806692.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.96394785  2.96394785  2.96394785 ... -0.32756018 -0.32756018
 -0.32756018]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\1711806692.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will r

In [27]:
train.head()

,temp,charge_rate,discharge_rate,CC_Kur,CC_Min,CC_Mean,cycle,capacity,id
0,-1.687651,2.963948,0,2.407742,2.769402,2.905546,0.0,3269.007095,25-025_1-#2
1,-1.687651,2.963948,0,2.460308,2.764711,2.887252,1.0,3269.019540,25-025_1-#2
2,-1.687651,2.963948,0,2.607434,2.817082,2.931155,2.0,3267.816991,25-025_1-#2
3,-1.687651,2.963948,0,2.637907,2.820242,2.930072,3.0,3265.782539,25-025_1-#2
4,-1.687651,2.963948,0,2.731444,2.828364,2.93264,4.0,3263.622593,25-025_1-#2


In [28]:
x_train, y_train = get_data(train)
x_test, y_test = get_data(test)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)
info = test.iloc[:, -3:]

(18210, 6) (18210,) (4040, 6) (4040,)


In [29]:
x_train.head()

,temp,charge_rate,discharge_rate,CC_Kur,CC_Min,CC_Mean
0,-1.687651,2.963948,0,2.407742,2.769402,2.905546
1,-1.687651,2.963948,0,2.460308,2.764711,2.887252
2,-1.687651,2.963948,0,2.607434,2.817082,2.931155
3,-1.687651,2.963948,0,2.637907,2.820242,2.930072
4,-1.687651,2.963948,0,2.731444,2.828364,2.93264


In [30]:
# RF
model = models['RF']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\RF.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_RF.csv', index=False)
# GBDT
model = models['GBDT']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\GBDT.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_GBDT.csv', index=False)


In [31]:
# AdaBoost
model = models['AdaBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\AdaBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_AdaBoost.csv', index=False)
# SVR
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
model = models['SVR']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\SVR.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_SVR.csv', index=False)
# NN
model = models['NN']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\NN.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred.flatten()})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_NN.csv', index=False)
# XGBoost
model = models['XGBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\XGBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CC\result_XGBoost.csv', index=False)


### CV阶段

In [32]:
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_CV.pkl', 'rb') as file:
    train, val, test = pickle.load(file)
train = pd.concat([train, val], axis=0).reset_index(drop=True)
train = add_feature(train)
test = add_feature(test)
# 归一化
scaler = StandardScaler()
train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
test.iloc[:, :-3] = scaler.transform(test.iloc[:, :-3])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\2882870631.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.68765121 -1.68765121 -1.68765121 ...  0.64009949  0.64009949
  0.64009949]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\2882870631.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.96394785  2.96394785  2.96394785 ... -0.32756018 -0.32756018
 -0.32756018]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
C:\Users\Administrator\AppData\Local\Temp\ipykernel_7284\2882870631.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

In [33]:
x_train, y_train = get_data(train)
x_test, y_test = get_data(test)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)
info = test.iloc[:, -3:]

(18210, 6) (18210,) (4040, 6) (4040,)


In [34]:
# RF
model = models['RF']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\RF.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_RF.csv', index=False)
# GBDT
model = models['GBDT']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\GBDT.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_GBDT.csv', index=False)


In [35]:
# AdaBoost
model = models['AdaBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\AdaBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_AdaBoost.csv', index=False)
# SVR
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
model = models['SVR']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\SVR.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_SVR.csv', index=False)
# NN
model = models['NN']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\NN.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred.flatten()})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_NN.csv', index=False)
# XGBoost
model = models['XGBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\XGBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\CV\result_XGBoost.csv', index=False)


### RE阶段

In [ ]:
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_RE.pkl', 'rb') as file:
    train, val, test = pickle.load(file)
train = pd.concat([train, val], axis=0).reset_index(drop=True)
train = add_feature(train)
test = add_feature(test)
# 归一化
scaler = StandardScaler()
train.iloc[:, :-3] = scaler.fit_transform(train.iloc[:, :-3])
test.iloc[:, :-3] = scaler.transform(test.iloc[:, :-3])

In [78]:
x_train, y_train = get_data(train)
x_test, y_test = get_data(test)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)
info = test.iloc[:, -3:]

(18210, 6) (18210,) (4040, 6) (4040,)


In [ ]:
# RF
model = models['RF']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\RF.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_RF.csv', index=False)
# GBDT
model = models['GBDT']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\GBDT.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_GBDT.csv', index=False)

In [ ]:
# AdaBoost
model = models['AdaBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\AdaBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_AdaBoost.csv', index=False)
# SVR
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
model = models['SVR']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\SVR.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_SVR.csv', index=False)
# NN
model = models['NN']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\NN.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred.flatten()})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_NN.csv', index=False)
# XGBoost
model = models['XGBoost']
model.fit(x_train, y_train)
# 保存模型
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\XGBoost.pkl', 'wb') as file:
    pickle.dump(model, file)
y_pred = model.predict(x_test)
result = pd.DataFrame({'real': y_test, 'pred': y_pred})
result = pd.concat([result, info], axis=1)
result.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\models\RE\result_XGBoost.csv', index=False)